# 🛠️ 实验 19：大模型工具调用 (Function Calling) 与 MCP 架构实战

在前面的检索增强生成 (RAG) 实验中，大模型只是通过**被动读取**我们预先加载好的检索片段来回答问题。这种方式下，大模型就像一个闭门读书的学者。

而在本实验中，我们将探讨如何让大模型成为一个**主动与外部世界交互的“智能体” (Agent)**。我们将具体学习五个模块：
0. **最原始的标签式工具调用原理**：使用特定的标签与 Python 的 `eval()` 实现最直观、手动的工具触发与执行；
1. **自定义 Tools 以及简单的 Function Calling**：学习大模型行业标准的 Function Calling 协议，理解大模型如何自动生成结构化 JSON 参数；
2. **MCP (Model Context Protocol) 架构与双端仿真**：掌握 Anthropic 提出的最新行业标准协议，理解 Client-Server 解耦的架构体系；
3. **基于 CLI 工具的文件检索 (CLI Search Tool)**：编写一个能够直接调用本地系统命令行（如 `grep`）的工具，展示大模型如何操纵底层操作系统完成任务；
4. **层级代理 (Hierarchical Agents) —— 将 LLM 作为工具进行调用**：了解如何让大模型调用另外一个具有专项 System Prompt 角色定义的大模型，完成协作处理。

---

## 🛠️ 环境初始化：云端 API 载入与 Mock 优雅降级机制

为了让云端 Function Calling 正常运行，我们将使用 OpenAI SDK 初始化大模型客户端。这里兼容了 **通义千问 (DashScope)** 与 **硅基流动 (SiliconFlow)** 两种主流 API 接入方式。

> 💡 **自适应降级设计**：如果您目前没有配置云端 API Key，模块一、模块三和模块四会自动激活 **Mock 模式**以确保 Notebook 可以顺利生成。不过，**模块零**为了展示真实的 ReAct 循环与 `eval()` 工具调用，**必须配置真实的 API Key 才能完整运行**。

In [ ]:
import os
import json
import time
import subprocess
from openai import OpenAI

# ============================================================
# 👇 请在下方引号内填入您的 API Key（通义千问或硅基流动二选一即可）
#    如已设置系统环境变量，程序将自动读取，无需在此重复填写。
# ============================================================
DASH_SCOPE_API_KEY = ""      # 例如: "sk-abc123..."
SILICONFLOW_API_KEY = ""     # 例如: "sk-abc123..."

openai_client = None
CLOUD_MODEL_NAME = ""

# 优先读取代码中手动输入的 Key，其次读取系统环境变量
api_key_dash = DASH_SCOPE_API_KEY or os.environ.get("DASH_SCOPE_API_KEY")
api_key_sf = SILICONFLOW_API_KEY or os.environ.get("SILICONFLOW_API_KEY")

if api_key_dash:
    openai_client = OpenAI(
        api_key=api_key_dash,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )
    CLOUD_MODEL_NAME = "qwen-plus" # 或 "qwen3.5-flash"
elif api_key_sf:
    openai_client = OpenAI(
        api_key=api_key_sf,
        base_url="https://api.siliconflow.cn/v1"
    )
    CLOUD_MODEL_NAME = "deepseek-ai/DeepSeek-V3"

if not openai_client:
    print("📢 [提示] 未检测到 API Key，模块 1、3、4 将以【本地 Mock 模拟模式】运行。")
    print("   如果您需要运行模块零，请在下方补充 API Key 或设置环境变量。")
else:
    print(f"✨ 成功载入 API 密钥！已实例化 OpenAI 客户端，模型设置为: {CLOUD_MODEL_NAME}")

---
## 🧱 模块零：用标签与 eval 实现最原始的工具调用（原理演示）

在 OpenAI 官方推出标准的 Function Calling 接口之前，早期的 AI 智能体（Agent）是如何调用工具的呢？

它的核心思想非常朴素：
1. **约定格式**：在 Prompt 中明确规定，如果大模型需要使用工具，必须输出特定格式的标签包裹的 Python 函数表达式，例如：`<tool>get_temperature('北京', '10月1日')</tool>`。
2. **文本检索与动态执行**：程序在后台通过字符串检索（或正则表达式）检测大模型的文本回复。如果发现标签，就用 Python 的 `eval()` 函数动态执行这段代码。
3. **结果替换与反馈**：将执行结果打包为 `<tool_output>...</tool_output>` 回馈给大模型，直到大模型认为收集齐了所有答案，直接输出自然语言回答。

> ⚠️ **安全警告**：在实际工程中，直接使用 `eval()` 运行大模型输出的不受控代码是**极其危险的**（存在代码注入漏洞，可能导致执行危险的系统删除指令）。本模块仅作基本原理解释和教学演示。

In [ ]:
# 1. 定义可供大模型调用的本地函数工具
def multiply(a, b):
    """返回 a 乘以 b"""
    return a * b

def divide(a, b):
    """返回 a 除以 b"""
    return a / b

def get_temperature(city: str, time: str) -> str:
    """返回 city 在 time 的气温"""
    # 模拟天气数据返回
    return f"{city}在 {time} 的温度是 22 度，秋高气爽。"

# 2. 系统 Prompt (System Prompt) 引导大模型如何输出工具调用标签
tool_instruction = """有必要可以使用工具，每一个工具都是函数。
使用工具的方式为输出 "<tool>[使用工具指令]</tool>"。
你会得到返回结果 "<tool_output>[工具返回的结果]</tool_output>"。
如果有使用工具的话，你应该告诉用户工具返回的结果。

可用工具：
multiply(a,b): 返回 a 乘以 b
divide(a,b): 返回 a 除以 b
get_temperature(city,time): 返回 city 在 time 的气温，注意 city 和 time 都是string类型"""

# 定义用户问题 (以中国大陆城市北京为例)
user_input = "告诉我北京 10月1日 天气如何啊？"
# 也可以测试数学计算问题：
# user_input = "请计算 111 * 222 / 777 是多少？"

messages = [
    {"role": "system", "content": tool_instruction},
    {"role": "user", "content": user_input}
]

# 3. ReAct 循环：大模型自主决定调用哪些工具，直到得出最终答案
if not openai_client:
    raise ValueError("📢 运行模块零需要配置云端大模型 API Key！请在【环境初始化】单元格中填写 Key 并运行。")

step = 0
while True:
    print(f"\n--- 🔄 第 {step+1} 轮大模型推理中... ---")
    
    # 呼叫大模型
    response = openai_client.chat.completions.create(
        model=CLOUD_MODEL_NAME,
        messages=messages,
        temperature=0.1 # 调低随机性，让标签输出更稳定
    )
    
    response_text = response.choices[0].message.content
    
    # 检测大模型是否输出工具调用标签
    if "</tool>" in response_text:
        # 提取第一个 <tool> 与 </tool> 之间的代码指令
        start_idx = response_text.find("<tool>") + len("<tool>")
        end_idx = response_text.find("</tool>")
        command = response_text[start_idx:end_idx].strip()
        
        print("🤖 大模型指令输出: ", response_text)
        print("⚙️ 提取出待调用的工具指令: ", command)
        
        # 使用 eval 动态执行本地函数
        try:
            tool_output = str(eval(command))
            print("💾 本地工具函数 eval 执行结果: ", tool_output)
        except Exception as e:
            tool_output = f"工具执行报错: {str(e)}"
            print("❌ 工具执行报错: ", e)
            
        # 裁剪模型回复：只保留到 </tool> 的部分
        clean_response = response_text.split("</tool>")[0] + "</tool>"
        
        # 将大模型表示“要使用工具”的这一轮对话追加到 messages 历史中
        messages.append({"role": "assistant", "content": clean_response})
        
        # 将本地执行得到的工具结果包裹成 <tool_output> 标签，作为 user 角色发送回大模型
        user_feedback = f"<tool_output>{tool_output}</tool_output>"
        messages.append({"role": "user", "content": user_feedback})
        
    else:
        # 模型没有输出工具调用标签，意味着大模型已经利用收集到的数据完成了回答
        print("🏆 大模型最终自然语言答复:")
        print("=" * 65)
        print(response_text)
        print("=" * 65)
        break
        
    step += 1
    if step > 5:
        print("⚠️ 思考轮数达到上限，防止死循环。")
        break

---
## 🧱 模块一：自定义 Tools 以及标准的 Function Calling

### 💡 为什么需要工业标准的 Function Calling？
通过模块零的演示，我们可以发现手工解析文本标签有几个明显痛点：
1. **格式不稳固**：大模型生成文本是概率性的，一旦少输了一个括号或写错了标签，程序解析就会失败。
2. **安全风险高**：直接使用 `eval()` 会带来灾难性的安全隐患。

因此，工业界制定了标准的 **Function Calling 协议**。大模型接收特定的 `tools` 参数描述（基于 JSON Schema 规范），并在需要时返回一个结构化的 `tool_calls` 参数对象，大模型本身只做**参数提取与提取决策**，彻底摆脱了 `eval` 的隐忧，同时大幅提升了多工具调用的格式稳定性。

下面，我们以查询“北京、上海、广州、深圳”的实时天气为例，演示标准的 Function Calling 流程。

### 1. 编写本地 Python 业务函数
这里我们写一个常规的 Python 业务函数 `get_weather`。它将在本模块中作为标准的 Tool 被大模型调用。

In [ ]:
def get_weather(city: str) -> str:
    """
    获取指定城市的实时天气数据
    """
    weather_db = {
        "北京": "晴，26°C，南风2级",
        "上海": "小雨，122°C，东风3级",
        "广州": "多云，30°C，无持续风向",
        "深圳": "阵雨，29°C，微风"
    }
    
    # 规范化城市名称后查询
    city_cleaned = city.replace("市", "").strip()
    result = weather_db.get(city_cleaned)
    if result:
        return f"{city}当前的实时天气为: {result}。"
    else:
        return f"抱歉，数据库中暂未收录 {city} 的天气数据，只支持北京、上海、广州和深圳。"

### 2. 声明大模型能看懂的 Tool Schema 定义
为了让大模型知道我们有什么工具、怎么调用，我们需要按照 JSON Schema 格式编写一份工具描述表。这份描述会被作为参数传给大模型。

In [ ]:
my_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "实时天气查询工具。当用户询问特定城市的天气、气温、风向等实时信息时使用此工具。",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "需要查询天气的城市名称，例如：北京、上海、广州。"
                    }
                },
                "required": ["city"]
            }
        }
    }
]
print("✅ 成功定义 1 个自定义 Tool Schema。")

### 3. 第一阶段：发送用户提问，让大模型决策并生成 Tool Call 参数
我们向大模型提出问题：“我想问一下上海今天天气怎么样？出门用带伞吗？”。由于我们在调用时提供了 `my_tools`，大模型会分析问题并发现需要调用 `get_weather` 工具。

In [ ]:
user_query = "我想问一下上海今天天气怎么样？出门用带伞吗？"
messages = [{"role": "user", "content": user_query}]

# 声明一个变量保存大模型做出的 Tool Call 决策
tool_calls = None
assistant_message = None

if openai_client:
    # 1. 真实云端调用
    response = openai_client.chat.completions.create(
        model=CLOUD_MODEL_NAME,
        messages=messages,
        tools=my_tools,      # 传入工具库描述
        tool_choice="auto"   # 让大模型自适应选择是否用工具
    )
    assistant_message = response.choices[0].message
    tool_calls = assistant_message.tool_calls
else:
    # 2. 本地 Mock 模式仿真
    print("⚠️ [MOCK 模式] 正在仿真大模型对工具调用的决策...")
    class MockFunction:
        def __init__(self, name, arguments):
            self.name = name
            self.arguments = arguments
            
    class MockToolCall:
        def __init__(self, id, name, arguments):
            self.id = id
            self.type = "function"
            self.function = MockFunction(name, arguments)
            
    class MockAssistantMessage:
        def __init__(self, content, tool_calls):
            self.role = "assistant"
            self.content = content
            self.tool_calls = tool_calls
            
    # 仿真模型输出：决定调用 get_weather 并生成参数 '{"city": "上海"}'
    tool_calls = [MockToolCall(
        id="call_weather_sf_123",
        name="get_weather",
        arguments='{"city": "上海"}'
    )]
    assistant_message = MockAssistantMessage(content=None, tool_calls=tool_calls)

# 观察大模型第一阶段的输出结构
print("🤖 大模型原始响应:")
if tool_calls:
    print(f"   - 决策结果: 【需要调用外部工具】")
    for index, call in enumerate(tool_calls):
        print(f"   - Tool Call [{index}]:")
        print(f"     ├─ ID: {call.id}")
        print(f"     ├─ 函数名: {call.function.name}")
        print(f"     └─ 参数 JSON: {call.function.arguments}")
else:
    print(f"   - 决策结果: 【直接回答，无需工具】")
    print(f"     内容: {assistant_message.content}")

### 3. 第二阶段：本地执行 Python 函数并将其反馈给大模型
大模型给出了它需要的工具和参数，但它本身无法真正调用 Python。我们将获取其参数，在本地执行 `get_weather` 函数，然后把结果包装成 `role: "tool"` 的消息再发送给大模型，获取最终包含天气事实的自然语言回答。

In [ ]:
# 把大模型的第一次回复追加到消息历史中 (必须原样传回，包括 tool_calls 结构)
messages.append(assistant_message)

if tool_calls:
    print("⚙️ 本地程序开始自动调度对应的 Python 函数...")
    for tool_call in tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        if func_name == "get_weather":
            # 1. 抓取参数并在本地执行
            city_param = func_args.get("city")
            local_result = get_weather(city_param)
            print(f"   ▶️ 成功触发本地 `get_weather(city='{city_param}')` 函数，执行结果为: '{local_result}'")
            
            # 2. 将执行结果作为专门的 tool 角色消息追加进历史
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,  # 必须与刚才的 Call ID 严格对齐
                "name": func_name,
                "content": local_result        # 反馈的执行结果
            })
            
    # 3. 将包含“问题 + 第一次决定调用的指示 + 函数反馈结果”的完整历史发送给大模型
    print("\n📤 正在将本地结果追加回 Context，请求大模型生成最终总结...")
    if openai_client:
        final_response = openai_client.chat.completions.create(
            model=CLOUD_MODEL_NAME,
            messages=messages
        )
        final_answer = final_response.choices[0].message.content
    else:
        time.sleep(0.5)
        final_answer = "根据最新的实时天气数据显示，上海今天有小雨，气温在 22°C 左右，伴有东风 3 级。因为今天有降雨，建议您出门时务必携带一把雨伞防雨。"
        
    print("\n🤖 大模型结合工具结果的最终回答:")
    print("=" * 60)
    print(final_answer)
    print("=" * 60)
else:
    print("无需执行工具，直接结束。")

---
## 🔌 模块二：MCP 的结构以及如何使用 MCP

### 💡 什么是 MCP (Model Context Protocol)？
在上面的模块一中，我们会发现一个问题：**大模型与本地函数的绑定是“紧耦合”的**。如果要在不同的项目里让大模型使用同一套天气查询、数据库操作、或者代码搜索的工具，你需要把这套 Python 函数和 JSON Schema 定义在每个项目里重写一遍。一旦工具升级，所有项目都要改写。

为此，Anthropic 推出了 **MCP (Model Context Protocol, 模型上下文协议)** 行业规范。它的核心目标是：**将大模型客户端与具体的工具服务解耦**。

```
 ┌─────────────────┐                 JSON-RPC 消息                ┌────────────────┐
 │  MCP Client     │ <──────────────────────────────────────────> │   MCP Server   │
 │                 │  (标准方法: list_tools, call_tool, 等)      │                │
 └────────┬────────┘                                              └────────┬───────┘
          │                                                                │
          │ 加载到                                                          │ 连接
          ▼                                                                ▼
   ┌─────────────┐                                                 ┌───────────────┐
   │ 大模型 (LLM) │                                                 │ 数据库/操作系统/│
   └─────────────┘                                                 │ 各种物理工具   │
                                                                   └───────────────┘
```

### 🔑 MCP 的三大核心资源要素
MCP 服务端主要向客户端暴露三样东西：
1. **Resources (资源)**：静态只读的数据流。可以是本地文本文件、数据库查询视图或日志信息。例如 `file:///workspace/logs/app.log`。
2. **Prompts (提示词模板)**：预设的 Prompt 快捷方式或对话模版，类似于“指令中心”，客户端可以直接拉取使用。
3. **Tools (工具)**：可供客户端请求执行的动态函数，也是 Function Calling 的核心载体。

### 🚀 双端通信底层：JSON-RPC 仿真实验
为了让大家在不用启动外部服务器、配置复杂网络和 Docker 的情况下直观看到 MCP 客户端与服务端的交互细节，我们将使用纯 Python 代码模拟一个 **JSON-RPC 通信协议仿真器**。在真实环境中，客户端与服务端正是通过标准输入输出 (Stdio) 或网络推送 (SSE) 来传输这类 JSON 数据的。

### 1. 编写 MCP Mock 服务端
服务端定义了可以被外部读取的资源 `course://syllabus` (大纲) 和工具 `calculate_grade` (成绩计算器)，并支持 JSON-RPC 格式的方法接收和返回。

In [ ]:
class MockMCPServer:
    """
    模拟一个 MCP 服务端，维护静态资源并提供动态工具支持，通过 JSON-RPC 与客户端交互。
    """
    def __init__(self):
        # 模拟服务端的【Resources (资源)】：学生大纲和打分标准
        self.resources = {
            "course://syllabus": "《人工智能基础》课程教学大纲：本课程涵盖线性回归、深度学习、Transformer、RAG 与 Function Calling 等模块。",
            "course://grading": "评分标准：期末笔试 50%，平时实验与作业 40%，出勤与课堂表现 10%。"
        }
        # 模拟服务端的【Tools (工具)】的 Schema
        self.tools = {
            "calculate_grade": {
                "description": "总评成绩计算器。输入期末考试成绩和平时实验作业成绩，按教学规定权重计算出总分。",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "exam_score": {"type": "number", "description": "期末笔试成绩 (百分制)"},
                        "lab_score": {"type": "number", "description": "平时实验作业成绩 (百分制)"}
                    },
                    "required": ["exam_score", "lab_score"]
                }
            }
        }
        
    def handle_message(self, request_str: str) -> str:
        """
        接收并解析符合 JSON-RPC 2.0 规范的请求消息，执行对应逻辑后返回 JSON-RPC 响应消息。
        """
        request = json.loads(request_str)
        msg_id = request.get("id")
        method = request.get("method")
        params = request.get("params", {})
        
        print(f"📥 [Server 收到消息] -> ID: {msg_id} | 请求方法: {method} | 携带参数: {params}")
        time.sleep(0.1)
        
        # 1. 对应 MCP 协议：获取资源列表
        if method == "resources/list":
            result = {
                "resources": [
                    {"uri": uri, "name": "教学大纲" if "syllabus" in uri else "评分标准", "mimeType": "text/plain"}
                    for uri in self.resources.keys()
                ]
            }
            return self._make_response(msg_id, result)
            
        # 2. 对应 MCP 协议：读取具体资源内容
        elif method == "resources/read":
            uri = params.get("uri")
            if uri in self.resources:
                result = {
                    "contents": [
                        {"uri": uri, "mimeType": "text/plain", "text": self.resources[uri]}
                    ]
                }
                return self._make_response(msg_id, result)
            else:
                return self._make_error(msg_id, 404, f"资源未找到: {uri}")
                
        # 3. 对应 MCP 协议：列出所有支持的工具
        elif method == "tools/list":
            result = {
                "tools": [
                    {"name": name, "description": info["description"], "inputSchema": info["parameters"]}
                    for name, info in self.tools.items()
                ]
            }
            return self._make_response(msg_id, result)
            
        # 4. 对应 MCP 协议：触发并调用特定的工具
        elif method == "tools/call":
            tool_name = params.get("name")
            tool_args = params.get("arguments", {})
            
            if tool_name == "calculate_grade":
                exam = float(tool_args.get("exam_score", 0))
                lab = float(tool_args.get("lab_score", 0))
                # 计算公式：笔试 50% + 实验 40% + 课堂表现默认送 10% 里的满分 (例如平时出勤折合10分)
                final_score = exam * 0.5 + lab * 0.4 + 10.0
                
                result = {
                    "content": [
                        {
                            "type": "text",
                            "text": f"【计算器运行成功】：期末成绩占比50%（折合{exam*0.5}分），实验成绩占比40%（折合{lab*0.4}分），平时出勤固定送10分。最终总评得分为: {final_score:.1f}分。"
                        }
                    ]
                }
                return self._make_response(msg_id, result)
            else:
                return self._make_error(msg_id, 404, f"未找到名为 {tool_name} 的工具")
                
        else:
            return self._make_error(msg_id, -32601, f"未实现的方法: {method}")
            
    def _make_response(self, msg_id, result):
        return json.dumps({
            "jsonrpc": "2.0",
            "id": msg_id,
            "result": result
        }, ensure_ascii=False)
        
    def _make_error(self, msg_id, code, message):
        return json.dumps({
            "jsonrpc": "2.0",
            "id": msg_id,
            "error": {"code": code, "message": message}
        }, ensure_ascii=False)

print("✅ MCP Mock Server 模块类编译完毕。")

### 2. 编写 MCP Mock 客户端并展示交互报文
客户端通过构造标准的 JSON-RPC 请求，通过 Stdio (此处模拟为直接的方法传值) 与服务端发生通信。请仔细观察打印出的发送和接收数据结构。

In [ ]:
class MockMCPClient:
    """
    模拟一个 MCP 客户端，通过发送规范化的 JSON-RPC 消息与服务端交换数据。
    """
    def __init__(self, server: MockMCPServer):
        self.server = server
        self.request_id = 0
        
    def request(self, method: str, params: dict = None) -> dict:
        self.request_id += 1
        # 构造标准的 JSON-RPC 2.0 报文
        payload = {
            "jsonrpc": "2.0",
            "id": self.request_id,
            "method": method,
            "params": params or {}
        }
        request_json = json.dumps(payload, ensure_ascii=False)
        print(f"📤 [Client 发送请求] -> 方法: {method} | 报文: {request_json}")
        
        # 发送到服务端，并获取服务端返回的报文
        response_json = self.server.handle_message(request_json)
        response = json.loads(response_json)
        
        # 解析处理结果
        print(f"📥 [Client 收到响应] -> ID: {response.get('id')}")
        if "error" in response:
            print(f"   ❌ 发生错误: {response['error']['message']}")
        else:
            print(f"   ✅ 成功返回: {response['result']}")
        print("-" * 80)
        return response

# 实例化双端
mcp_server = MockMCPServer()
mcp_client = MockMCPClient(mcp_server)

print("✨ 开始模拟 MCP 协议标准交互流程：\n")

# 步骤 1: 客户端发现服务端的静态资源列表
resources_list = mcp_client.request("resources/list")

# 步骤 2: 客户端读取具体的静态资源（例如获取教学大纲）
mcp_client.request("resources/read", {"uri": "course://syllabus"})

# 步骤 3: 客户端发现服务端可用的动态工具列表
tools_list = mcp_client.request("tools/list")

# 步骤 4: 客户端触发工具，要求计算平时分 80，期末笔试 90 的学生总评成绩
mcp_client.request("tools/call", {
    "name": "calculate_grade",
    "arguments": {
        "exam_score": 90,
        "lab_score": 80
    }
})

---
## 🔍 模块三：如何使用 CLI 工具完成简单的文件检索

### 💡 大模型与操作系统的联动 (OS World Interaction)
在实际工程项目中（比如各种智能编程助手、自动化运维 Agent），大模型不仅可以做简单的 API 查询，甚至还可以通过**执行系统 shell 命令行工具**（如 `ls`、`grep`、`find`）来查找和编辑本地文件。

在本模块中，我们将亲自编写一个能够调用操作系统底层的 **CLI 检索工具**，并且将其注册给大模型，让大模型可以通过执行命令行来检索课表信息数据库文件。

### 1. 在本地自动创建课程信息文本文档
我们将在 `data/` 文件夹下生成一个包含排课与教室安排的数据文本文档 `course_info.txt`。

In [ ]:
# 确保 data 目录存在
os.makedirs("data", exist_ok=True)

course_data = """课程编号: CS101 | 课程名称: 线性回归实验 | 上课教室: 综合楼 402 | 授课老师: 张教授
课程编号: CS102 | 课程名称: 深度学习与文字接龙 | 上课教室: 实验楼 301 | 授课老师: 李副教授
课程编号: CS103 | 课程名称: Transformer与注意力机制 | 上课教室: 信息楼 508 | 授课老师: 王讲师
课程编号: CS104 | 课程名称: 检索增强生成(RAG)实战 | 上课教室: 科技楼 204 | 授课老师: 赵教授
课程编号: CS105 | 课程名称: 人工智能大模型工具调用 | 上课教室: 实验楼 502 | 授课老师: 刘副教授
"""

txt_file_path = "data/course_info.txt"
with open(txt_file_path, "w", encoding="utf-8") as f:
    f.write(course_data.strip())

print(f"💾 课程原始文本文档已成功写入至: {txt_file_path}")

### 2. 编写基于 Python `subprocess` 模块的 CLI grep 检索函数
在 Python 中，我们要安全地触发系统命令，通常使用 `subprocess` 模块。为了防止路径和关键字注入的安全漏洞，我们应当避免拼接 Shell 字符串，而是使用列表形式将命令与参数分开传递。

In [ ]:
def run_cli_grep_search(keyword: str, file_path: str = "data/course_info.txt") -> str:
    """
    使用底层操作系统的 `grep` 命令在指定文件中进行文本搜索
    """
    print(f"🔧 [CLI 进程运行] 正在启动操作系统进程执行: grep '{keyword}' {file_path}")
    
    try:
        # 使用列表形式安全传递参数，禁用 shell=True 防御潜在的参数注入漏洞
        # capture_output=True 可以获取标准输出与标准错误
        process = subprocess.run(
            ["grep", keyword, file_path],
            capture_output=True,
            text=True,
            check=False
        )
        
        # grep 正常检索到内容返回码为 0
        if process.returncode == 0:
            return process.stdout.strip()
        # grep 没有找到任何匹配返回码为 1
        elif process.returncode == 1:
            return f"[CLI 结果] 未在文件 {file_path} 中检索到包含关键字 '{keyword}' 的课程信息。"
        else:
            return f"[CLI 错误] 进程执行出错: {process.stderr.strip()}"
            
    except FileNotFoundError:
        # 如果是 Windows 系统，默认没有 grep 命令，我们需要进行自适应降级兼容处理
        print("⚠️ [系统兼容提示] 当前系统未找到 grep 命令，切换到 Python 模拟的 grep 实现...")
        # 模拟 grep 功能
        try:
            matches = []
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    if keyword in line:
                        matches.append(line.strip())
            if matches:
                return "\n".join(matches)
            return f"[MOCK CLI 结果] 未在文件 {file_path} 中检索到包含关键字 '{keyword}' 的课程信息。"
        except Exception as inner_err:
            return f"[MOCK CLI 错误] 模拟检索失败: {str(inner_err)}"
            
    except Exception as e:
        return f"[运行时异常] 执行出错: {str(e)}"

# 验证工具可用性
print("🧪 验证本地 CLI grep 工具:")
print(run_cli_grep_search("Transformer"))

### 3. 将 CLI 工具注册给大模型，实现端到端 Agent 检索
我们将 `run_cli_grep_search` 的 Schema 注册给大模型，提问大模型关于排课的细节问题，并完整走通调用链路。

In [ ]:
# 1. 定义 CLI Tool 的 JSON Schema
cli_tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "run_cli_grep_search",
            "description": "在本地课表数据库 (data/course_info.txt) 中通过系统内置 grep 命令搜索含有指定关键字的排课文本。适用于查询某门课的上课教室、授课老师等场景。",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "需要进行 grep 检索的课程名或教师名等单词关键字，例如：'RAG'、'刘副教授'。"
                    }
                },
                "required": ["keyword"]
            }
        }
    }
]

# 2. 准备问题与对话上下文
agent_query = "赵教授今天在哪个教室上课？他的那门课程叫什么名字？"
agent_messages = [{"role": "user", "content": agent_query}]

agent_tool_calls = None
agent_assistant_msg = None

# 3. 第一阶段：发送给大模型决策
if openai_client:
    response = openai_client.chat.completions.create(
        model=CLOUD_MODEL_NAME,
        messages=agent_messages,
        tools=cli_tools_schema,
        tool_choice="auto"
    )
    agent_assistant_msg = response.choices[0].message
    agent_tool_calls = agent_assistant_msg.tool_calls
else:
    # 本地 Mock 模式模拟
    print("⚠️ [MOCK 模式] 模拟大模型提取出搜索关键字 '赵教授' 并生成 Tool Call...")
    class MockFunction:
        def __init__(self, name, arguments):
            self.name = name
            self.arguments = arguments
    class MockToolCall:
        def __init__(self, id, name, arguments):
            self.id = id
            self.type = "function"
            self.function = MockFunction(name, arguments)
    class MockMsg:
        def __init__(self):
            self.role = "assistant"
            self.content = None
            self.tool_calls = [MockToolCall("call_grep_099", "run_cli_grep_search", '{"keyword": "赵教授"}')]
    agent_assistant_msg = MockMsg()
    agent_tool_calls = agent_assistant_msg.tool_calls

# 4. 第二阶段：如果模型决策要调用，就在本地运行 CLI 并把输出塞给大模型
agent_messages.append(agent_assistant_msg)

if agent_tool_calls:
    for call in agent_tool_calls:
        func_name = call.function.name
        func_args = json.loads(call.function.arguments)
        
        if func_name == "run_cli_grep_search":
            kw = func_args.get("keyword")
            # 物理调用 grep 检索
            cli_output = run_cli_grep_search(keyword=kw)
            print(f"   📥 [CLI 执行反馈] grep 搜索到数据为:\n   {cli_output}")
            
            agent_messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": func_name,
                "content": cli_output
            })
            
    # 5. 请求大模型做自然语言汇总
    print("\n📤 正在请求大模型生成基于 CLI 搜索结果的自然语言回复...")
    if openai_client:
        final_agent_response = openai_client.chat.completions.create(
            model=CLOUD_MODEL_NAME,
            messages=agent_messages
        )
        agent_answer = final_agent_response.choices[0].message.content
    else:
        time.sleep(0.5)
        # 结合 `课程编号: CS104 | 课程名称: 检索增强生成(RAG)实战 | 上课教室: 科技楼 204 | 授课老师: 赵教授` 模拟回复
        agent_answer = "赵教授今天在科技楼 204 教室上课，他讲授的课程名称叫做《检索增强生成(RAG)实战》。"
        
    print("\n🤖 大模型最终回答:")
    print("=" * 60)
    print(agent_answer)
    print("=" * 60)
else:
    print("大模型未决定调用 CLI 工具。直接回复:", agent_assistant_msg.content)

---
## 🤖 模块四：层级代理 (Hierarchical Agents) —— 将 LLM 作为工具进行调用

在前面的实验中，我们的工具都是常规的 Python 代码或系统命令行工具。但在实际的复杂 Agent 系统（如多智能体系统 Multi-Agent System）中，**大模型本身也可以作为另一个大模型的工具来使用**。

### 💡 为什么需要将 LLM 作为工具？
1. **职责分离 (Specialization)**：主大模型（Orchestrator，类似于项目经理）只负责拆解规划任务与结果汇总。具体的子任务（如专业翻译、SQL 生成、情感分析）通过工具调用分摊给配置了特定 System Prompt 的“专职子大模型”（Worker LLM）。
2. **上下文隔离**：当需要做海量文本摘要或翻译时，通过子模型调用可以实现长文本处理隔离，避免主大模型的上下文窗口过载。

本模块中，我们将演示：主大模型接收到一个“生成日常问候语并翻译成日语”的任务，主模型在规划后，自动触发我们写好的“专业翻译子大模型”工具来完成工作。

In [ ]:
# 1. 定义一个在内部调用 LLM 的 Python 工具函数
def llm_translator_tool(text: str, target_language: str) -> str:
    """
    大模型专属翻译工具。当用户需要将指定的文本翻译成英语、日语、法语等其他语言时，调用此工具。
    """
    print(f"🔧 [LLM Tool 运行] 内部专门翻译子 Agent 启动。目标语言: {target_language} | 待译文本: '{text}'")
    
    # 构造子大模型的专属翻译 System Prompt，使其聚焦于翻译，过滤其他杂音
    sub_system_prompt = (
        f"你是一个资深的专业翻译官，精通中外互译。请将用户输入的文本精确地翻译成{target_language}。\n"
        "要求：保持原文语气，仅输出翻译后的文本本身，绝对不要输出任何多余的解释、前言或额外声明。"
    )
    
    if openai_client:
        try:
            # 启动子 LLM 运行
            sub_response = openai_client.chat.completions.create(
                model=CLOUD_MODEL_NAME,
                messages=[
                    {"role": "system", "content": sub_system_prompt},
                    {"role": "user", "content": text}
                ],
                temperature=0.3  # 翻译任务调低温度，使其输出更稳定
            )
            translation_result = sub_response.choices[0].message.content.strip()
            return translation_result
        except Exception as e:
            return f"[内部子 LLM 报错] 翻译失败: {str(e)}"
    else:
        # Mock 模式下的仿真翻译返回
        time.sleep(0.5)
        mock_translations = {
            "英文": "Hello, wish you have a wonderful day today!",
            "英语": "Hello, wish you have a wonderful day today!",
            "日语": "こんにちは、今日一日が楽しいものでありますように！",
            "法语": "Bonjour, je vous souhaite une excellente journée aujourd'hui!"
        }
        return mock_translations.get(target_language, f"[MOCK TRANSLATION to {target_language}]: {text}")

# 2. 验证本地 LLM 翻译工具可用性
print("🧪 验证本地内置的 LLM 翻译工具工作情况:")
print(llm_translator_tool("你好，祝你今天过得愉快！", "日语"))

In [ ]:
# 3. 定义大模型专用的工具描述 Schema
llm_tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "llm_translator_tool",
            "description": "多语言翻译工具。当用户要求翻译某段话、将某句话变成其他语言时调用此工具。",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "需要翻译的原始文本内容。"
                    },
                    "target_language": {
                        "type": "string",
                        "description": "目标语言，例如：中文、英语、日语、法语。"
                    }
                },
                "required": ["text", "target_language"]
            }
        }
    }
]

# 4. 提出一个涉及“规划任务”与“调用子翻译任务”的问题给主大模型
main_query = "请帮我写一句中文的日常问候语，然后把它翻译成日语。"
print(f"👤 用户提问: '{main_query}'\n")

main_messages = [{"role": "user", "content": main_query}]

main_tool_calls = None
main_assistant_msg = None

# 第一阶段：主大模型接收任务，自主生成一句中文，并决策需要翻译
if openai_client:
    response = openai_client.chat.completions.create(
        model=CLOUD_MODEL_NAME,
        messages=main_messages,
        tools=llm_tools_schema,
        tool_choice="auto"
    )
    main_assistant_msg = response.choices[0].message
    main_tool_calls = main_assistant_msg.tool_calls
else:
    # 本地 Mock 模式仿真
    print("⚠️ [MOCK 模式] 模拟主大模型思考生成中文，并决定触发子翻译 Agent...")
    class MockFunction:
        def __init__(self, name, arguments):
            self.name = name
            self.arguments = arguments
    class MockToolCall:
        def __init__(self, id, name, arguments):
            self.id = id
            self.type = "function"
            self.function = MockFunction(name, arguments)
    class MockMsg:
        def __init__(self):
            self.role = "assistant"
            self.content = None
            self.tool_calls = [MockToolCall(
                "call_llm_translate_001", 
                "llm_translator_tool", 
                '{"text": "你好，祝你今天过得愉快！", "target_language": "日语"}'
            )]
    main_assistant_msg = MockMsg()
    main_tool_calls = main_assistant_msg.tool_calls

# 第二阶段：触发本地的“翻译工具”（调用内部子大模型完成翻译）
main_messages.append(main_assistant_msg)

if main_tool_calls:
    for call in main_tool_calls:
        func_name = call.function.name
        func_args = json.loads(call.function.arguments)
        
        if func_name == "llm_translator_tool":
            txt_to_trans = func_args.get("text")
            lang = func_args.get("target_language")
            
            # 触发翻译子模型，获取其翻译文本
            translation_output = llm_translator_tool(text=txt_to_trans, target_language=lang)
            print(f"   📥 [子 LLM 运行反馈] 翻译结果为: '{translation_output}'")
            
            main_messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": func_name,
                "content": translation_output
            })
            
    # 第三阶段：主大模型整合问候语与翻译结果，输出最终回复给用户
    print("\n📤 正在请求主大模型整合结果...")
    if openai_client:
        final_main_response = openai_client.chat.completions.create(
            model=CLOUD_MODEL_NAME,
            messages=main_messages
        )
        final_answer = final_main_response.choices[0].message.content
    else:
        time.sleep(0.5)
        final_answer = (
            "为您编写的中文日常问候语是：『你好，祝你今天过得愉快！』\n"
            "调用专业翻译子模型处理后，对应的日语翻译为：『こんにちは、今日一日が楽しいものでありますように！』"
        )
        
    print("\n🤖 主大模型整合后的最终回答:")
    print("=" * 65)
    print(final_answer)
    print("=" * 65)
else:
    print("主大模型直接回答:", main_assistant_msg.content)

---
## 📈 课后知识复盘与思考练习

通过本实验，我们共同学习了大模型对外交互的核心原理：
1. **Function Calling 的本质**：大模型并不执行代码，它只负责做**逻辑决策（决定用什么工具）**与**参数提取（将非结构化文本转化为结构化 JSON 参数）**。执行动作由我们的 Python 宿主程序代劳。
2. **MCP (Model Context Protocol) 核心价值**：它是解耦 Client（大模型前端/IDE）与 Server（包含具体天气 API、数据库连通器、文件检索器的服务）的标准交互协议，基于 JSON-RPC 消息报文交互资源与工具服务。
3. **CLI 命令行联动的 Agent**：通过 `subprocess` 调用 `grep`/`find` 等系统命令行，展现了 LLM 操纵底层操作系统文件的强大边界。这种思路是构建全自动 AI 程序员、运维机器人的核心设计模式。
4. **LLM 作为工具 (LLM-as-a-Tool)**：在工具函数内部启动另一个大模型（层级代理）。通过为子模型配置专门的 System Prompt（如翻译、SQL 编写），实现职责解耦，降低主模型的上下文压力。

### 🧠 思考题
在模块三的 `run_cli_grep_search` 函数中，我们有意识地将 Python 的 `subprocess.run` 参数列表分拆传递（即 `["grep", keyword, file_path]`），而不是直接运行拼接后的字符串 `shell=True` (比如执行 `subprocess.run(f"grep {keyword} {file_path}", shell=True)`)。这是出于什么安全设计考量？如果用户不怀好意地提问，可能引发什么严重的后果？

*(提示：如果用户提问：“我想检索关键字：'RAG; rm -rf /'，会发生什么？”)*